# Data Processing Pipeline

Orchestration notebook for the SemEval2026 Task 13A data processing pipeline.

**Steps:**
1. Load & Clean raw data
2. Outlier flagging
3. Save processed data
4. Feature extraction (handcrafted)
5. Build validation splits
6. Sanity checks

In [6]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import yaml
from pathlib import Path

# Load project config
with open('../configs/global.yaml') as f:
    cfg = yaml.safe_load(f)

PATHS = cfg['paths']
SPLITS = cfg['splits']
SEED = cfg['seed']

# Resolve relative paths from project root
ROOT = Path('..')
for k, v in PATHS.items():
    PATHS[k] = str(ROOT / v)
for k, v in SPLITS.items():
    SPLITS[k] = str(ROOT / v)

print('Config loaded.')
print(f'Seed: {SEED}')
print(f'Train: {PATHS["train"]}')
print(f'Val:   {PATHS["validation"]}')
print(f'Test:  {PATHS["test"]}')

Config loaded.
Seed: 42
Train: ..\data\raw\train.parquet
Val:   ..\data\raw\validation.parquet
Test:  ..\data\raw\test.parquet


## Step 1: Load & Clean

In [7]:
from src.data.preprocess import load_raw_data, preprocess_dataframe

train_df, val_df, test_df = load_raw_data(
    PATHS['train'], PATHS['test'], PATHS['validation']
)

print(f'Raw train: {train_df.shape}')
print(f'Raw val:   {val_df.shape}')
print(f'Raw test:  {test_df.shape}')

# Clean and add basic length features
train_df = preprocess_dataframe(train_df)
val_df = preprocess_dataframe(val_df)
test_df = preprocess_dataframe(test_df)

print(f'\nAfter preprocessing:')
print(f'Train columns: {train_df.columns.tolist()}')
print(f'Train shape: {train_df.shape}')

Raw train: (500000, 4)
Raw val:   (100000, 4)
Raw test:  (500000, 2)

After preprocessing:
Train columns: ['code', 'generator', 'label', 'language', 'code_length', 'num_lines']
Train shape: (500000, 6)


## Step 2: Outlier Flagging

In [8]:
from src.data.preprocess import flag_outliers

# Flag outliers on all sets
train_df = flag_outliers(train_df)
val_df = flag_outliers(val_df)
test_df = flag_outliers(test_df)

Outlier flags: 5,169 / 500,000 rows flagged (empty=1, binary=103, minified=68, extreme_len=4999)
Outlier flags: 1,033 / 100,000 rows flagged (empty=0, binary=27, minified=6, extreme_len=1000)
Outlier flags: 5,018 / 500,000 rows flagged (empty=0, binary=11, minified=8, extreme_len=4999)


In [9]:
# Inspect flagged samples
for flag in ['is_empty', 'is_binary_ish', 'is_minified']:
    flagged = train_df[train_df[flag]]
    if len(flagged) > 0:
        print(f'\n=== {flag}: {len(flagged)} samples ===')
        for _, row in flagged.head(3).iterrows():
            print(f'  len={row["code_length"]:,} | label={row.get("label", "?")} | code[:100]={repr(row["code"][:100])}')
    else:
        print(f'{flag}: 0 samples')


=== is_empty: 1 samples ===
  len=0 | label=0 | code[:100]=''

=== is_binary_ish: 103 samples ===
  len=446 | label=1 | code[:100]='a = list(map(int, input().split()))\n\n\xa0\xa0\xa0#print(list(map(int, input().split())))\nfor i in range(n):\n\xa0'
  len=561 | label=1 | code[:100]='# code starts here\ndef factorial(n):\n\xa0\xa0\xa0\xa0factor = 1\n\n\xa0\xa0\xa0\xa0for i in range(2, n+1):\n\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0factor *= i'
  len=342 | label=1 | code[:100]='def wordSubsets(self, A: List[str], B: List[str]) -> List[str]:\n\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0def check(s1,s2):\n\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0'

=== is_minified: 68 samples ===
  len=644 | label=1 | code[:100]='Tags: `Adobe Premiere Pro` `Word of the Day` `Cyberpunk` `CrystalGraphics` `Ashland Street Food Truc'
  len=518 | label=1 | code[:100]='In this solution, the function `game_winner` takes a string `s` as input and checks if the length of'
  len=700 | label=1 | code[:100]='In this code, we first create

In [10]:
# Drop only truly pathological samples (empty + binary-ish)
pathological_mask = train_df['is_empty'] | train_df['is_binary_ish']
n_pathological = pathological_mask.sum()
if n_pathological > 0:
    train_df = train_df[~pathological_mask].copy()
    print(f'Dropped {n_pathological} pathological samples (empty/binary).')
else:
    print('No pathological samples to drop.')

print(f'Final train size: {len(train_df):,}')

Dropped 104 pathological samples (empty/binary).
Final train size: 499,896


## Step 3: Save Processed Data

In [11]:
# Save processed dataframes
processed_dir = ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(processed_dir / 'train_clean.parquet', index=False)
val_df.to_parquet(processed_dir / 'val_clean.parquet', index=False)
test_df.to_parquet(processed_dir / 'test_clean.parquet', index=False)

print(f'Saved to {processed_dir}/')
print(f'  train_clean.parquet: {len(train_df):,} rows')
print(f'  val_clean.parquet:   {len(val_df):,} rows')
print(f'  test_clean.parquet:  {len(test_df):,} rows')

Saved to ..\data\processed/
  train_clean.parquet: 499,896 rows
  val_clean.parquet:   100,000 rows
  test_clean.parquet:  500,000 rows


## Step 4: Feature Extraction

In [12]:
from src.data.preprocess import extract_all_features

print('Extracting features for train...')
train_features = extract_all_features(train_df)
print(f'Train features: {train_features.shape}')

print('\nExtracting features for val...')
val_features = extract_all_features(val_df)
print(f'Val features: {val_features.shape}')

print('\nExtracting features for test...')
test_features = extract_all_features(test_df)
print(f'Test features: {test_features.shape}')

Extracting features for train...


100%|██████████| 499896/499896 [04:50<00:00, 1719.43it/s]


Train features: (499896, 40)

Extracting features for val...


100%|██████████| 100000/100000 [01:00<00:00, 1663.73it/s]


Val features: (100000, 40)

Extracting features for test...


100%|██████████| 500000/500000 [07:08<00:00, 1166.05it/s]


Test features: (500000, 40)


In [13]:
# Save features
features_dir = ROOT / 'data' / 'features'
features_dir.mkdir(parents=True, exist_ok=True)

train_features.to_parquet(features_dir / 'handcrafted_train.parquet', index=False)
val_features.to_parquet(features_dir / 'handcrafted_val.parquet', index=False)
test_features.to_parquet(features_dir / 'handcrafted_test.parquet', index=False)

print(f'Saved features to {features_dir}/')
print(f'Feature columns ({len(train_features.columns)}): {train_features.columns.tolist()}')

Saved features to ..\data\features/
Feature columns (40): ['num_lines', 'num_non_empty_lines', 'blank_line_ratio', 'mean_line_length', 'max_line_length', 'std_line_length', 'whitespace_ratio', 'tab_ratio', 'mean_indent', 'max_indent', 'std_indent', 'char_count', 'alpha_ratio', 'digit_ratio', 'special_char_ratio', 'uppercase_ratio', 'num_functions', 'num_classes', 'num_imports', 'num_loops', 'num_conditionals', 'num_returns', 'num_try_catch', 'brace_count', 'paren_count', 'bracket_count', 'semicolon_count', 'comma_count', 'dot_count', 'colon_count', 'operator_count', 'comment_line_count', 'comment_density', 'max_nesting_depth', 'mean_indent_depth', 'max_indent_depth', 'num_identifiers', 'mean_identifier_length', 'std_identifier_length', 'unique_identifier_ratio']


## Step 5: Build Validation Splits

Strategy:
- **Self-splits from train** for daily ablation (OOD-first)
- **Official `validation.parquet`** as lockbox / sanity check
- All splits stratified by **label x language**

In [14]:
from src.data.build_splits import (
    build_holdout_split,
    build_random_stratified,
    build_lolo_language,
    build_length_style_bins,
    save_splits,
)

# Reset index for clean splits
train_for_splits = train_df.reset_index(drop=True)

# 1. Single holdout split (90/10)
holdout = build_holdout_split(train_for_splits, seed=SEED)
save_splits(holdout, SPLITS['holdout'])
print(f'Holdout split: train={len(holdout["holdout"]["train"]):,}, val={len(holdout["holdout"]["val"]):,}')

# 2. Random stratified k-fold
kfold = build_random_stratified(train_for_splits, n_splits=10, seed=SEED)
save_splits(kfold, SPLITS['random'])
fold0 = kfold['fold_0']
print(f'K-fold (10): fold_0 train={len(fold0["train"]):,}, val={len(fold0["val"]):,}')

# 3. Leave-one-language-out
lolo = build_lolo_language(train_for_splits)
save_splits(lolo, SPLITS['proxy_ood_language'])
for name, split in lolo.items():
    print(f'LOLO {name}: train={len(split["train"]):,}, val={len(split["val"]):,}')

# 4. Length/style bins
bins = build_length_style_bins(train_for_splits)
save_splits(bins, SPLITS['proxy_ood_style'])
for name, split in bins.items():
    print(f'Bins {name}: train={len(split["train"]):,}, val={len(split["val"]):,}')

Holdout split: train=449,906, val=49,990
K-fold (10): fold_0 train=449,906, val=49,990
LOLO holdout_Python: train=42,677, val=457,219
LOLO holdout_Java: train=480,605, val=19,291
LOLO holdout_C++: train=476,510, val=23,386
Bins holdout_bin_0: train=333,011, val=166,885
Bins holdout_bin_2: train=333,324, val=166,572
Bins holdout_bin_1: train=333,457, val=166,439


## Step 6: Sanity Checks

In [15]:
# Verify output files exist
print('=== Output Files ===')
for d in ['data/processed', 'data/features', 'data/splits']:
    p = ROOT / d
    if p.exists():
        files = sorted(p.iterdir())
        for f in files:
            size_mb = f.stat().st_size / 1024 / 1024
            print(f'  {f.relative_to(ROOT)}  ({size_mb:.1f} MB)')
    else:
        print(f'  {d}/ NOT FOUND')

=== Output Files ===
  data\processed\test_clean.parquet  (281.3 MB)
  data\processed\train_clean.parquet  (193.0 MB)
  data\processed\val_clean.parquet  (38.5 MB)
  data\features\handcrafted_test.parquet  (44.8 MB)
  data\features\handcrafted_train.parquet  (39.2 MB)
  data\features\handcrafted_val.parquet  (8.7 MB)
  data\splits\holdout_v1.json  (7.0 MB)
  data\splits\length_style_bins_v1.json  (21.1 MB)
  data\splits\lolo_language_v1.json  (21.1 MB)
  data\splits\random_stratified_v1.json  (70.5 MB)


In [16]:
# Verify stratification in holdout split
train_split_idx = holdout['holdout']['train']
val_split_idx = holdout['holdout']['val']

train_split = train_for_splits.iloc[train_split_idx]
val_split = train_for_splits.iloc[val_split_idx]

print('=== Holdout Split Stratification ===')
print(f'\nAI ratio:  train={train_split["label"].mean():.4f}  val={val_split["label"].mean():.4f}')
print(f'\nLanguage proportions:')
for lang in sorted(train_for_splits['language'].unique()):
    t_pct = (train_split['language'] == lang).mean() * 100
    v_pct = (val_split['language'] == lang).mean() * 100
    print(f'  {lang:8s}:  train={t_pct:.2f}%  val={v_pct:.2f}%')

print('\nPer-language AI ratio:')
for lang in sorted(train_for_splits['language'].unique()):
    t_ratio = train_split[train_split['language'] == lang]['label'].mean()
    v_ratio = val_split[val_split['language'] == lang]['label'].mean()
    print(f'  {lang:8s}:  train={t_ratio:.4f}  val={v_ratio:.4f}')

=== Holdout Split Stratification ===

AI ratio:  train=0.5230  val=0.5230

Language proportions:
  C++     :  train=4.68%  val=4.68%
  Java    :  train=3.86%  val=3.86%
  Python  :  train=91.46%  val=91.46%

Per-language AI ratio:
  C++     :  train=0.5234  val=0.5233
  Java    :  train=0.5218  val=0.5220
  Python  :  train=0.5230  val=0.5230


In [17]:
print('\n=== Pipeline Complete ===')
print(f'Processed train: {len(train_df):,} rows')
print(f'Val (lockbox):   {len(val_df):,} rows')
print(f'Test (blind):    {len(test_df):,} rows')
print(f'Features:        {len(train_features.columns)} columns')
print(f'Splits:          holdout + 10-fold + LOLO + length-bins')


=== Pipeline Complete ===
Processed train: 499,896 rows
Val (lockbox):   100,000 rows
Test (blind):    500,000 rows
Features:        40 columns
Splits:          holdout + 10-fold + LOLO + length-bins
